# RAG - Simple Version

A complete RAG system in about 60 lines of code.

This is the same thing as the live class notebook, but simpler. Every cell runs from top to bottom.

---

## The architecture

Real RAG systems have two separate halves. Keep them separate in your head - they run at different times and they fail for different reasons.

```
   INGEST  (runs once, when documents change)

   documents  ->  chunk  ->  embed  ->  store in ChromaDB


   QUERY  (runs every time someone asks something)

   question   ->  embed  ->  search ChromaDB  ->  put chunks in prompt  ->  GPT answers
```

That is the whole system. Everything below is those six boxes.

---

## What changed from the live class version

**One dictionary per chunk instead of four separate lists.**

The live class kept `chunk_texts`, `chunk_titles`, `ids` and `metadatas` as four separate lists that had to stay the same length. When one of them was left empty, ChromaDB crashed with `Non-empty lists are required for ['metadatas']`.

Here each chunk is a single dictionary:

```python
{"id": "...", "title": "...", "text": "..."}
```

Parallel lists cannot get out of sync if there is only one list. This is how you would write it at work.

**One `search()` function that can also filter**, instead of two near-identical functions.

**A `chunk_document()` that returns a plain list**, so there is no tuple unpacking to get wrong.

## Setup

In [ ]:
!pip install -q openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

In [ ]:
import openai
from getpass import getpass

# getpass shows a box. Paste your key there.
# Nothing is saved in the file, so the notebook is safe to share.
openai.api_key = getpass("Paste your OpenAI API key: ")

CHAT_MODEL = "gpt-4o-mini"                 # writes the answers
EMBEDDING_MODEL = "text-embedding-3-small" # turns text into numbers

print("Ready")

## Step 1 - Why we need RAG

Ask the model about our own internal document. It has never seen it.

In [ ]:
question = "In the HireAI MVP plan, which database is used and why?"

response = openai.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": question}]
)

print(response.choices[0].message.content)

The model either admits it does not know, or it invents a confident answer. Inventing is called **hallucination** - the model produces a *plausible* answer, not a *true* one, and it cannot tell you which one you just got.

RAG fixes this by finding the right part of our document and putting it into the prompt first.

## Step 2 - The documents

Each document is a **dictionary** with a title and its text.

Keeping the title next to the text - instead of in a separate `titles` list - is the single change that removes most of the bugs from this notebook.

In [ ]:
documents = [
    {
        "title": "HireAI Architecture Plan",
        "text": """# HireAI — Scalable MVP Architecture Plan (Free-Tier First)

**Goal:** Ship an MVP for the first 100–1,000 users using free AI tools and free-tier infrastructure, built with Claude Code — architected so that migrating to the full paid/scaled stack (the one in `HireAI-Architecture.md`) is a config swap, not a rewrite.

---

## 1. The Core Principle: Modular Monolith + Adapter Boundaries

Do **not** build microservices, Kubernetes, or Kong for 1,000 users. Build **one deployable app** whose internal folders are the exact service boundaries you'll split along later. Every external dependency (LLM, storage, queue, telephony, email) sits behind a small interface ("adapter"), so scaling = swapping the adapter's implementation via an environment variable.

```
apps/
  web/                     → Next.js 15 (App Router) — dashboard + candidate portal + API
packages/
  db/                      → Drizzle ORM schema + migrations (Postgres)
  core/                    → business logic, organized by module (future services)
    jobs/                  → JD generation, job CRUD
    candidates/            → resume parsing, scoring, ranking
    screening/             → Round-1 screening (async text/voice)
    scheduling/            → slots, bookings, calendar sync
    interviews/            → AI interview engine (Round 2)
    analytics/             → pipeline funnel, reports
  adapters/                → ALL external dependencies behind interfaces
    llm/                   → generate(), extract(), score(), embed()
    storage/               → put(), get(), signedUrl()        (S3-compatible API)
    queue/                 → enqueue(), worker()               (pg-boss now, BullMQ later)
    email/                 → send()
    telephony/             → call(), transcribe()              (stub in MVP)
  shared/                  → types, zod schemas, auth helpers
```

**Rule that makes migration painless:** `core/*` modules never import each other's internals and never call vendors directly — only through `adapters/*` and the DB. When you outgrow the monolith, each `core/` folder becomes a service with the same interface, and each adapter gets a production implementation.

---

## 2. MVP Tech Stack (all free tiers)

| Layer | MVP (free) | Scale target (from HireAI-Architecture.md) | Migration effort |
|---|---|---|---|
| Frontend + API | Next.js 15 + TypeScript + Tailwind + shadcn/ui on **Vercel free** | Same frontend; API extracted to NestJS/Fastify pods | Low — `core/` is framework-agnostic |
| Database | **Supabase free** (Postgres 500MB + pgvector) | RDS / Supabase Pro Postgres | None — same engine, same Drizzle migrations |
| ORM | Drizzle (SQL-first, no runtime engine) | Same | None |
| Auth | **Supabase Auth** (email + Google OAuth, free) | Same, or JWT/OAuth2 service | None until enterprise SSO |
| Vector search | **pgvector** in same Postgres | pgvector on bigger instance (Pinecone only if >10M vectors) | None |
| File storage | **Supabase Storage** (1GB free, S3-compatible) | AWS S3 | Env-var swap — same S3 API in adapter |
| Background jobs | **pg-boss** (queue inside Postgres — zero extra infra) | Redis + BullMQ | Adapter swap; job payloads identical |
| Cache | None needed at this scale (Postgres is fast enough) | Redis / Upstash | Add later behind adapter |
| Full-text search | Postgres `tsvector` | Elasticsearch/Typesense if needed | Deferred; likely never needed |
| Realtime (live transcript) | Supabase Realtime channels | Socket.io / managed WS | Adapter swap |
| Email | Resend free (3k/mo) or Brevo free (300/day) | SendGrid/SES | Adapter swap |
| Monitoring | Sentry free + Vercel analytics | Sentry + Datadog + PostHog | Additive |
| Billing | Skip in MVP (manual invoices) | Stripe | Additive module |

**Why Supabase as the spine:** one free account gives Postgres + pgvector + Auth + Storage + Realtime — the entire data layer of the target architecture, with a direct paid upgrade path and no vendor rewrite (it's plain Postgres and S3 APIs underneath).

---

## 3. Free AI Stack (the important part)

All AI calls go through **one adapter**: `adapters/llm` exposing `generate()`, `extractStructured()`, `scoreAgainstRubric()`, `embed()`. Provider chosen by env var, with automatic fallback chain when a free tier rate-limits.

| Capability | Free option (MVP) | Notes / limits | Scale swap |
|---|---|---|---|
| JD generation, scoring, question gen | **Google Gemini 2.5 Flash** free tier | ~1,500 req/day free — plenty for 1k users | Claude Sonnet/Opus via API key swap |
| Fallback LLM | **Groq** (Llama 3.3 70B) free tier | Very fast; good for scoring/extraction | Same |
| Structured resume extraction | Gemini Flash with JSON schema output | pdf-parse / mammoth for text extraction first (free, local) | Claude with tool-use |
| Embeddings (candidate↔JD matching) | **Gemini `text-embedding-004`** (free) or local `bge-small` via transformers.js | Stored in pgvector | OpenAI/Voyage embeddings |
| Speech-to-text | **Groq Whisper large-v3** (free tier) | For uploaded audio answers | OpenAI Whisper / Deepgram |
| Text-to-speech | **Edge-TTS** (free, unofficial MS voices) or browser SpeechSynthesis | Good enough for async interviews | ElevenLabs / OpenAI TTS |
| Dev-time AI | **Claude Code** | Builds/refactors the codebase itself | — |

**Rate-limit strategy:** every LLM call goes through the queue with retry + provider fallback (Gemini → Groq → defer). At 1,000 users this stays inside free tiers because the heavy AI work (parsing, scoring) is per-candidate, batched, and cached (never re-score an unchanged resume — hash the resume+JD pair).

---

## 4. The One Big MVP Substitution: Voice → Async

Twilio outbound IVR calls are the most expensive, most complex, least-free part of the target architecture. For the MVP, **replace live phone screening with an async web-based screening**, which keeps the exact same data model (`screening_calls` table: transcript, scores, recording URL):

- **Round 1 (MVP):** Candidate gets an email/WhatsApp link → opens a mobile web page → answers 5–8 questions by **recording audio in the browser** (MediaRecorder API, free) → uploads to storage → queue job runs Groq Whisper STT → LLM scores against rubric → auto-advance past threshold.
- **Round 2 (MVP):** Same page, but conversational: TTS asks a question (Edge-TTS/browser), candidate answers by voice, STT + LLM generate an adaptive follow-up. This is the full "AI interview" experience with zero telephony cost.
- **Scale:** implement `adapters/telephony` with Twilio + streaming STT — the screening module's inputs/outputs (transcript in, scores out) don't change at all.

This is the design decision that makes a free MVP possible without repainting yourself into a corner.

---

## 5. Data Model (multi-tenant from day 1)

Use the schema already defined in `HireAI-Architecture.md` §5 almost verbatim — it's good. Non-negotiables to bake in now because they're brutal to retrofit:

1. **`org_id` on every table** + Postgres **Row-Level Security** (Supabase makes this native). Multi-tenancy is the #1 thing you cannot bolt on later.
2. **UUIDs for all primary keys** (no sequential int leaks, safe for future sharding).
3. **Status fields as explicit state machines** (`candidate.status`: `uploaded → parsed → scored → screening_invited → screening_done → interview_scheduled → interview_done → shortlisted / rejected`). The whole pipeline automation hangs off these transitions.
4. **`events` append-only table** (org_id, entity, entity_id, event_type, payload, created_at) — this gives you analytics, audit trail, and debugging for free, and becomes your event bus when you later move to real queues/services.
5. **All AI outputs stored as `jsonb` with the prompt version** (`ai_scores: {version: "v3", ...}`) so you can re-score when prompts improve and audit for EEOC/bias compliance.

---

## 6. Phase Plan (each phase ships usable value)

**Phase 0 — Foundation (week 1)**
Monorepo (pnpm + Turborepo), Next.js app, Supabase project, Drizzle schema + RLS, Supabase Auth (email + Google), org/user onboarding, CI via GitHub Actions (free), Sentry. Deploy to Vercel on day 1 — deploy continuously from here.

**Phase 1 — Jobs + JD Generation (week 2)**
Job CRUD, LLM adapter with Gemini/Groq fallback, JD generator with templates, skills extraction into structured fields. *First demoable value.*

**Phase 2 — Resume Pipeline (weeks 3–4)**
Bulk upload (drag-drop → Supabase Storage), pg-boss queue, text extraction (pdf-parse/mammoth), LLM structured extraction, embedding + pgvector similarity, weighted scoring vs JD, ranked candidate list with threshold filter. *This is the core product — most of the recommendation-engine work lives here.*

**Phase 3 — Async Screening, Round 1 (weeks 5–6)**
Candidate-facing mobile web page, browser audio recording, Whisper STT via Groq, rubric scoring, auto-advance on threshold, recruiter review UI with playback + transcript.

**Phase 4 — Scheduling (week 7)**
Recruiter availability slots, candidate self-serve booking page, Google Calendar API (free), email notifications via Resend. Skip Outlook until a customer asks.

**Phase 5 — AI Interview, Round 2 (weeks 8–9)**
Conversational async interview: TTS question → voice answer → adaptive follow-up, dimensional scoring, report generation. Reuses everything from Phase 3.

**Phase 6 — Pipeline Dashboard + Handoff (week 10)**
Funnel view per job, candidate comparison, one-click Round-3 scheduling, org analytics off the `events` table. Then: beta users.

**Deferred until revenue:** Twilio live IVR, Outlook/MS Graph, Stripe billing, Elasticsearch, Redis, SOC 2 tooling.

---

## 7. Migration Playbook (MVP → Scale, "starting tomorrow")

Triggers and the exact move for each — no rewrites, because of the adapter boundaries:

| Pain signal | Move | Why it's cheap |
|---|---|---|
| Free LLM rate limits hit | Set `LLM_PROVIDER=anthropic`, add API key | Same adapter interface |
| Supabase 500MB DB full | Upgrade Supabase plan or `pg_dump` → RDS | Plain Postgres |
| Queue jobs backing up | Swap pg-boss impl → BullMQ + Upstash Redis in `adapters/queue` | Same job payloads |
| Storage >1GB | Point storage adapter at S3 | Same S3 API |
| Vercel function timeouts on long AI jobs | Move worker process to Railway/Fly.io container (queue consumer already runs standalone) | Worker is already a separate entrypoint |
| Need live phone screening (paying customers) | Implement `adapters/telephony` with Twilio; screening module unchanged | Transcript-in/scores-out contract |
| API load / team growth | Extract `core/candidates` (heaviest) into its own service first; `events` table becomes the integration point | Module boundaries = service boundaries |
| Enterprise deals | Add Stripe, SSO (SAML via WorkOS), SOC 2 logging on the `events` table | Additive |

**Monthly cost: $0 for the MVP phase.** First paid line items, in order, will be: LLM API (~$50–200), Supabase Pro ($25), a worker container (~$5–10).

---

## 8. Claude Code Working Method

- Keep this file + a `CLAUDE.md` (conventions: adapter rule, RLS rule, state-machine statuses) at repo root so every session has the architecture in context.
- Build phase by phase; each phase = one milestone with its own branch and deployed preview.
- Have Claude Code write integration tests for each `core/` module against a local Postgres (Docker) — these tests are your safety net for every adapter swap during migration.
- Use zod schemas in `packages/shared` as the single source of truth for API contracts — when modules become services, the contracts already exist.
""",
    },
    # add more documents here, same shape
]


print("Documents:", len(documents))
print("Characters:", len(documents[0]["text"]))

## Step 3 - Chunking

An 11,000 character document is too big to search inside, and too big to paste into every prompt. So we cut it into small pieces called **chunks** of about 700 characters.

We start a new chunk in two situations:

1. We hit a Markdown heading (`#`) - the author is telling us a new topic starts
2. The current chunk is full

Each chunk comes back as a dictionary with an `id`, a `title` and the `text`.

> **The bug from the live class was here.** The `chunks.append(...)` and `return chunks` lines were indented *inside* the `for` loop, so the function returned after reading the very first line and gave you 1 chunk. Look at the indentation of the last block below - it is outside the loop. Python never warns you about this; the code just runs and gives the wrong answer.

In [ ]:
def chunk_document(doc, max_size=700):
    """Cut one document into chunks. Returns a list of dictionaries."""

    chunks = []
    section = "Introduction"    # the heading we are currently under
    buffer = ""                 # text collected so far for the current chunk

    for line in doc["text"].split("\n"):

        is_heading = line.startswith("#")
        is_full = len(buffer) + len(line) > max_size

        # save the chunk we have collected, if there is one
        if buffer.strip() != "" and (is_heading or is_full):
            chunks.append({
                "id": doc["title"] + " #" + str(len(chunks)),
                "title": doc["title"] + " > " + section,
                "text": buffer.strip(),
            })
            buffer = ""

        if is_heading:
            section = line.replace("#", "").strip()
        else:
            buffer = buffer + line + "\n"

    # save whatever is left over.
    # NOTE: this block is OUTSIDE the for loop. That was the live class bug.
    if buffer.strip() != "":
        chunks.append({
            "id": doc["title"] + " #" + str(len(chunks)),
            "title": doc["title"] + " > " + section,
            "text": buffer.strip(),
        })

    return chunks

In [ ]:
# Chunk every document into one flat list
chunks = []
for doc in documents:
    chunks = chunks + chunk_document(doc)

print("Total chunks:", len(chunks))
print()

for c in chunks:
    print(len(c["text"]), "chars |", c["title"])

In [ ]:
# Always read a few of your own chunks before going further.
# Most retrieval problems are visible right here.

print(chunks[0]["text"])
print()
print("-" * 60)
print()
print(chunks[7]["text"])

## Step 4 - Embeddings

A computer cannot compare two sentences for **meaning**. It can compare numbers very well.

An **embedding** is a list of numbers representing the meaning of a text. The useful property:

> Two texts with a similar meaning get similar numbers, even when they share no words.

We send many texts in **one API call** rather than one call per text. That is faster, cheaper, and how you would do it in production.

In [ ]:
def embed_texts(texts):
    """Turn a list of texts into a list of embeddings, in one API call."""
    response = openai.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


test = embed_texts(["How do I roll back a deployment?"])

print("An embedding is a list of", len(test[0]), "numbers")
print("First 8:", test[0][:8])

In [ ]:
import numpy as np

def similarity(a, b):
    """Cosine similarity. 1 = same meaning, 0 = unrelated."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


sentences = [
    "How do I roll back a deployment?",          # compare everything to this
    "What is the process to revert a release?",  # same meaning, no shared words
    "Undo the last deploy of the service.",      # related
    "How do I apply for annual leave?",          # different topic
    "The capital of France is Paris.",           # unrelated
]

vectors = embed_texts(sentences)     # all five in one call

for i in range(len(sentences)):
    print(round(similarity(vectors[0], vectors[i]), 3), "|", sentences[i])

Sentence 2 shares no important word with sentence 1 - "roll back" vs "revert", "deployment" vs "release" - and still scores high.

A keyword search would score that pair zero. This is the whole reason RAG works.

## Step 5 - Store the chunks in ChromaDB

ChromaDB is a vector database that runs inside the notebook. No server, no account.

It stores four things per chunk:

| Field | What it is | Why |
|---|---|---|
| `ids` | unique name | so you can update or delete it later |
| `embeddings` | the numbers | this is what the search compares |
| `documents` | the text | to read it and put it in the prompt |
| `metadatas` | extra info | to show the source, and to filter |

All four lists must be the same length. Because we build all four from the same `chunks` list in one place, **they cannot get out of sync** - which is exactly the bug that broke the live class.

In [ ]:
import chromadb

chroma_client = chromadb.Client()   # in memory. Use PersistentClient(path="./db") to keep it.

# delete first so this cell can be run again without an error
try:
    chroma_client.delete_collection("docs")
except:
    pass

collection = chroma_client.create_collection(
    name="docs",
    metadata={"hnsw:space": "cosine"}   # cosine similarity, same as Step 4
)

# All four lists come from the same source, so they always match.
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embed_texts([c["text"] for c in chunks]),
    documents=[c["text"] for c in chunks],
    metadatas=[{"title": c["title"]} for c in chunks],
)

print("Stored", collection.count(), "chunks")

## Step 6 - Search

Embed the question, then ask ChromaDB for the closest chunks.

Two things to know:

- ChromaDB returns a **distance**, not a similarity. Smaller is closer, so we convert with `1 - distance`.
- It returns lists inside lists, because you can search several questions at once. We only send one, so we always take `[0]`.

The optional `where` argument filters on metadata **during** the search. One function handles both plain search and filtered search.

In [ ]:
def search(question, n=3, where=None):
    """Find the n chunks closest in meaning to the question."""

    results = collection.query(
        query_embeddings=embed_texts([question]),
        n_results=n,
        where=where,          # e.g. {"title": "..."} to search one section only
    )

    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "title": results["metadatas"][0][i]["title"],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
        })
    return hits


for hit in search("Which database should the MVP use and why?"):
    print(round(hit["score"], 3), "|", hit["title"])
    print("   ", hit["text"][:120].replace("\n", " "), "...")
    print()

In [ ]:
# Notice: none of these questions use the same words as the document,
# and the right section still comes back.

for q in ["how do we handle phone calls in the MVP?",
          "what happens when the free AI tier runs out?",
          "what must we get right in the database from day one?"]:

    print("Q:", q)
    for hit in search(q, n=2):
        print("   ", round(hit["score"], 3), "|", hit["title"])
    print()

## Step 7 - Answer the question

Now put the retrieved chunks into a prompt.

Three rules in the prompt do most of the work:

1. **Only use the documents given.** Stops the model falling back on training memory.
2. **Cite the source number.** Lets the reader verify.
3. **Say "I don't know" when the documents do not cover it.**

Rule 3 is the most important line in this notebook. A model will not refuse unless you explicitly allow it to.

In [ ]:
SYSTEM_PROMPT = """You answer questions about internal technical documents.

Rules:
1. Answer ONLY using the documents below. Do not use any other knowledge.
2. Write the source number after each fact, like this: [1]
3. If the documents do not contain the answer, reply exactly:
   "I don't know based on the documents provided."
4. Be short and clear."""


def ask(question, n=3):
    """Full RAG: search, build the prompt, get the answer."""

    hits = search(question, n=n)

    # number the sources so the model can cite them
    sources = []
    for i, hit in enumerate(hits, start=1):
        sources.append("[" + str(i) + "] " + hit["title"] + "\n" + hit["text"])

    user_prompt = "DOCUMENTS:\n" + "\n\n".join(sources) + "\n\nQUESTION: " + question

    response = openai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,      # same answer every time
    )

    answer = response.choices[0].message.content

    print(answer)
    print("\nSOURCES:")
    for i, hit in enumerate(hits, start=1):
        print(" ", "[" + str(i) + "]", hit["title"], "|", round(hit["score"], 3))
    print("\nTokens:", response.usage.total_tokens)

    return answer

In [ ]:
ask("Which database should the MVP use, and why was it chosen?")

In [ ]:
ask("How is voice screening handled in the MVP instead of Twilio calls?")

### The test that matters most

Ask something the document does not cover. A good RAG system refuses. A bad one invents.

This single test decides whether you can put the system in front of a real user.

In [ ]:
ask("What is our company's deploy window for production?")

**Try it:** delete rule 3 from `SYSTEM_PROMPT`, re-run that cell and this one. The model will invent a deploy window.

## Step 8 - Filtering

`where` narrows the search to matching metadata **before** ranking, so other chunks are never considered.

In a real product this is how access control works: store `user_id` in the metadata and filter on it, so a user physically cannot retrieve someone else's documents.

In [ ]:
section = "HireAI Architecture Plan > 2. MVP Tech Stack (all free tiers)"

print("Whole document:")
for hit in search("what do we use for storage?", n=3):
    print(" ", round(hit["score"], 3), "|", hit["title"])

print("\nOnly the tech stack section:")
for hit in search("what do we use for storage?", n=2, where={"title": section}):
    print(" ", round(hit["score"], 3), "|", hit["title"])

## Step 9 - Chat

Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if "exit" in user_input.lower():
        break
    print()
    ask(user_input)
    print()

## Step 10 - Use your own documents

Replace the `documents` list in Step 2, then re-run Step 3 onwards. Nothing else changes.

In [ ]:
# --- text file (upload it with the folder icon in Colab) ---
# with open("/content/my_file.txt") as f:
#     documents = [{"title": "My File", "text": f.read()}]

# --- PDF ---
# !pip install -q pypdf
# from pypdf import PdfReader
# reader = PdfReader("/content/my_report.pdf")
# text = "\n\n".join(page.extract_text() for page in reader.pages)
# documents = [{"title": "My Report", "text": text}]
#
# A PDF has no # headings, so every chunk will be under "Introduction".
# That is fine - it still cuts by size.

# --- several documents ---
# documents = [
#     {"title": "Architecture Plan", "text": text_1},
#     {"title": "Security Policy",   "text": text_2},
# ]

print("Uncomment whichever you need, then re-run from Step 3")

## Summary

The whole system, in the order it runs:

| Step | Code | What it does |
|---|---|---|
| 2 | `documents` | a list of `{"title", "text"}` dictionaries |
| 3 | `chunk_document()` | cuts each document into `{"id", "title", "text"}` chunks |
| 4 | `embed_texts()` | turns a list of texts into a list of number-vectors |
| 5 | `collection.add()` | stores ids, embeddings, text and metadata together |
| 6 | `search()` | embeds the question, returns the closest chunks |
| 7 | `ask()` | puts those chunks in a prompt, gets a cited answer |

### Five things to remember

1. **RAG = search + a model that writes.** Most RAG problems are search problems.
2. **One dictionary per chunk, not parallel lists.** Lists that must stay the same length will eventually not.
3. **Embeddings match meaning, not words.** That is why "revert a release" finds "rollback".
4. **Always allow "I don't know".** Without that rule the model invents.
5. **Never put an API key in a notebook you share.**

### When an answer is wrong, it is one of three things

| Cause | How to check | Fix |
|---|---|---|
| The right text was never in a chunk | print your chunks and read them | chunking |
| It was in a chunk but search missed it | run `search()` on its own | raise `n`, better chunks |
| Search found it, answer still wrong | the right chunk IS in the sources list | the prompt |

**Run `search()` on its own first.** Five seconds, and it tells you whether the problem is retrieval or the model. Most people skip this and spend an hour rewriting the prompt when the chunking was wrong.

---

### What a production version adds

You now have the core. Real systems add these on top, in roughly this order:

| Add | Why |
|---|---|
| Keyword search (BM25) alongside embeddings | embeddings are weak on exact codes and IDs like `ERR_5012` |
| A reranker | a second, more accurate pass over the top 20 candidates |
| A saved test set of questions + expected sources | so you can prove a change helped instead of guessing |
| `PersistentClient` and re-indexing only changed files | not rebuilding everything for a one-page edit |
| Logging the retrieved chunk ids per question | so "it gave a wrong answer yesterday" is debuggable |

None of them change the six boxes at the top. They make each box better.

## Homework

1. **Break it on purpose.** Indent the last `if` block in `chunk_document()` into the `for` loop. Re-run Step 3. You should get 1 chunk. Then fix it.
2. Change `max_size` from 700 to 150, re-run Steps 3 and 5, then ask a question. Better or worse? Why?
3. Change `n=3` to `n=1` in `ask()` and try the data model question. What is missing?
4. Delete rule 3 from `SYSTEM_PROMPT` and ask about the deploy window again.
5. Add a second document to the list and ask a question that needs both.